# Notebook 04 — Modelo Global de Regressão Linear

Este notebook responde à pergunta central de pesquisa do trabalho:

> Quais linguagens de programação estão associadas a salários mais altos
> para desenvolvedores, controlando por experiência, país e formação?

A resposta é construída por meio de um modelo de regressão linear múltipla (OLS) estimado sobre a amostra de desenvolvedores em sentido estrito preparada nos notebooks anteriores. As escolhas metodológicas (baselines
USA / backend / Bacharelado, top 20 linguagens em formato dummy) seguem o
que foi decidido na análise exploratória.

O notebook cobre todos os instrumentos analíticos exigidos: identificação
de variáveis, estimação do modelo, R² e ajuste, análise de resíduos,
ANOVA e previsões.

In [1]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Modelagem estatística
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.anova import anova_lm

# Testes estatísticos
from scipy import stats

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais consistentes com notebooks anteriores
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# Carrega o dataset preparado no notebook 02. Este arquivo já contém:
# - Filtros aplicados (devs profissionais, salário não-nulo, etc.)
# - Outliers tratados (faixa USD 1.000 a USD 500.000)
# - Colunas categóricas agrupadas (Country_agrupado, DevType_agrupado, EdLevel_agrupado)
# - 20 dummies de linguagens com prefixo lang_
# - log_salario para uso em modelo de robustez
df = pd.read_csv("../data/processed/df_limpo.csv")

print(f"Linhas:   {len(df):,}")
print(f"Colunas:  {df.shape[1]}")

Linhas:   17,712
Colunas:  33


In [3]:
# Antes de qualquer coisa, confirmamos que as colunas que o modelo precisa
# realmente existem no dataset carregado. Esta verificação evita o erro
# clássico de "KeyError" a 50 linhas de código depois.

colunas_esperadas = {
    "salario": "ConvertedCompYearly",
    "log_salario": "log_salario",
    "experiencia": "YearsCode",
    "pais_agrupado": "Country_agrupado",
    "devtype_agrupado": "DevType_agrupado",
    "edlevel_agrupado": "EdLevel_agrupado",
}

# Verificação das colunas-chave
faltando = [v for v in colunas_esperadas.values() if v not in df.columns]
assert not faltando, f"Colunas ausentes no dataset: {faltando}"

# Verificação das 20 dummies de linguagem
dummies_lang = [c for c in df.columns if c.startswith("lang_")]
assert len(dummies_lang) == 20, f"Esperadas 20 dummies, encontradas {len(dummies_lang)}"

print("Todas as colunas necessárias estão presentes.")
print(f"Dummies de linguagem encontradas ({len(dummies_lang)}):")
print(", ".join(sorted(dummies_lang)))

Todas as colunas necessárias estão presentes.
Dummies de linguagem encontradas (20):
lang_Bash_Shell, lang_C, lang_CPlusPlus, lang_CSharp, lang_Dart, lang_Go, lang_Groovy, lang_HTML_CSS, lang_Java, lang_JavaScript, lang_Kotlin, lang_Lua, lang_PHP, lang_PowerShell, lang_Python, lang_Ruby, lang_Rust, lang_SQL, lang_Swift, lang_TypeScript


In [4]:
# Inspeção visual rápida das primeiras linhas, restrita às colunas
# que efetivamente vão entrar no modelo. Isso confirma que os tipos
# estão corretos e que os valores fazem sentido.
colunas_modelo = [
    "ConvertedCompYearly",
    "YearsCode",
    "Country_agrupado",
    "DevType_agrupado",
    "EdLevel_agrupado",
] + sorted(dummies_lang)

df[colunas_modelo].head()

,ConvertedCompYearly,YearsCode,Country_agrupado,DevType_agrupado,EdLevel_agrupado,lang_Bash_Shell,lang_C,lang_CPlusPlus,lang_CSharp,lang_Dart,lang_Go,lang_Groovy,lang_HTML_CSS,lang_Java,lang_JavaScript,lang_Kotlin,lang_Lua,lang_PHP,lang_PowerShell,lang_Python,lang_Ruby,lang_Rust,lang_SQL,lang_Swift,lang_TypeScript
0,"61,256.0000",14.0000,Ukraine,mobile,Mestrado,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,"104,413.0000",10.0000,Netherlands,backend,Tecnólogo,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,"53,061.0000",12.0000,Ukraine,frontend,Bacharelado,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1
3,"36,197.0000",5.0000,Ukraine,backend,Bacharelado,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0
4,"60,000.0000",22.0000,Ukraine,outros,Mestrado,0,1,1,1,0,0,0,1,1,1,0,1,0,1,1,0,0,1,0,1


## 2. Identificação das variáveis

O modelo de regressão linear múltipla utiliza uma variável dependente
(o salário anual em dólares) e três grupos de variáveis explicativas:
um conjunto principal (linguagens de programação utilizadas) e dois
conjuntos de controle (experiência profissional e atributos do
respondente). Esta seção descreve o papel, o tipo e a escala de cada
variável, e justifica as categorias de referência adotadas para os
fatores qualitativos.

In [5]:
# Construção da tabela de identificação de variáveis.
# A tabela documenta o papel de cada variável no modelo, seu tipo
# estatístico e sua escala. Esta é a primeira evidência apresentada
# ao avaliador na seção de desenvolvimento do relatório.

variaveis = pd.DataFrame([
    # Variável dependente
    {
        "Variável": "ConvertedCompYearly",
        "Papel": "Dependente (Y)",
        "Tipo": "Quantitativa contínua",
        "Escala": "USD/ano",
        "Descrição": "Salário anual total convertido para dólares",
    },
    # Variáveis principais (X de interesse)
    {
        "Variável": "lang_* (20 dummies)",
        "Papel": "Explicativa principal",
        "Tipo": "Qualitativa binária",
        "Escala": "0 = não usa, 1 = usa",
        "Descrição": "Indicadores das 20 linguagens mais frequentes",
    },
    # Variáveis de controle
    {
        "Variável": "YearsCode",
        "Papel": "Controle",
        "Tipo": "Quantitativa discreta",
        "Escala": "Anos (1 a 50)",
        "Descrição": "Anos totais de experiência com programação",
    },
    {
        "Variável": "Country_agrupado",
        "Papel": "Controle",
        "Tipo": "Qualitativa nominal",
        "Escala": "11 categorias",
        "Descrição": "País de residência (top 10 + Outros). Referência: USA",
    },
    {
        "Variável": "DevType_agrupado",
        "Papel": "Controle",
        "Tipo": "Qualitativa nominal",
        "Escala": "6 categorias",
        "Descrição": "Área de atuação. Referência: backend",
    },
    {
        "Variável": "EdLevel_agrupado",
        "Papel": "Controle",
        "Tipo": "Qualitativa ordinal",
        "Escala": "8 categorias",
        "Descrição": "Maior nível de formação concluído. Referência: Bacharelado",
    },
])

variaveis

,Variável,Papel,Tipo,Escala,Descrição
0,ConvertedCompYearly,Dependente (Y),Quantitativa contínua,USD/ano,Salário anual total convertido para dólares
1,lang_* (20 dummies),Explicativa principal,Qualitativa binária,"0 = não usa, 1 = usa",Indicadores das 20 linguagens mais frequentes
2,YearsCode,Controle,Quantitativa discreta,Anos (1 a 50),Anos totais de experiência com programação
3,Country_agrupado,Controle,Qualitativa nominal,11 categorias,País de residência (top 10 + Outros). Referênc...
4,DevType_agrupado,Controle,Qualitativa nominal,6 categorias,Área de atuação. Referência: backend
5,EdLevel_agrupado,Controle,Qualitativa ordinal,8 categorias,Maior nível de formação concluído. Referência:...


In [6]:
# Persiste a tabela em CSV para uso no PDF de apresentação
variaveis.to_csv("../output/tables/04_identificacao_variaveis.csv", index=False)
print("Tabela salva em output/tables/04_identificacao_variaveis.csv")

Tabela salva em output/tables/04_identificacao_variaveis.csv


### 2.1 Categorias de referência

Variáveis qualitativas precisam de uma categoria de referência para que
o modelo seja identificável. A escolha não altera a qualidade do ajuste,
mas afeta diretamente a leitura dos coeficientes: cada coeficiente passa
a ser interpretado como a diferença salarial em relação à categoria
omitida.

As referências adotadas neste estudo foram:

- **País — Estados Unidos.** É o mercado mais representado na amostra
  (cerca de 25% dos respondentes) e o de maior salário mediano.
  Adotá-lo como referência produz coeficientes com leitura direta:
  valores negativos indicam quanto se ganha a menos, em média, em cada
  outro país, mantendo as demais variáveis constantes.

- **Área de atuação — back-end.** É a área técnica com maior número de
  respondentes na amostra após fullstack, o que estabiliza a estimação,
  e serve como ponto de comparação natural para áreas correlatas
  (frontend, fullstack, mobile).

- **Formação — Bacharelado.** Concentra cerca de 46% da amostra. Por
  ser a categoria modal, oferece a comparação mais informativa: os
  coeficientes das demais formações expressam o quanto a formação
  diverge do padrão dominante na profissão.

### 2.2 Forma funcional do modelo

A equação geral do modelo, antes da estimação dos coeficientes, é:

$$
\text{Salário}_i = \beta_0 + \sum_{k=1}^{20} \beta_k \cdot \text{lang}_{k,i}
+ \gamma \cdot \text{YearsCode}_i
+ \sum_{p=1}^{10} \delta_p \cdot \text{País}_{p,i}
+ \sum_{a=1}^{5} \theta_a \cdot \text{Área}_{a,i}
+ \sum_{f=1}^{7} \phi_f \cdot \text{Formação}_{f,i}
+ \varepsilon_i
$$

onde $\varepsilon_i$ representa o termo de erro do indivíduo $i$.

Cada somatório exclui a categoria de referência: o país de referência
(EUA) não tem dummy, a área de referência (back-end) não tem dummy, e a
formação de referência (Bacharelado) não tem dummy. Isso evita
colinearidade perfeita entre as dummies e o intercepto.